# L81 · 音乐推荐系统 — 用户喜好 → 嵌入向量 → k-NN 邻居 → 推荐列表

**目标**：实现完整推荐流程——给定用户喜欢的 N 首歌，计算用户 profile embedding，找最近邻，过滤已听歌曲，输出推荐列表。

🔗 **Aurora 连接**：本节是 Aurora Music Intelligence Engine 的核心交付物，对应 `src/aurora/music/recommend.py` 中的 `recommend()` 函数；所用 embedding 由 `src/aurora/music/embed.py` 产出，相似度检索由 `src/aurora/music/sim.py`（即 `mu3_sim`）提供。

## 核心直觉

把每首歌看作向量空间中的一个点，用户的偏好就是这些点的**重心**（算术平均）。离重心最近的歌曲，在语义上与用户历史口味最一致——这是协同过滤思想在 embedding 空间的几何翻译，简单却在工业界被反复验证有效。


In [ ]:
import numpy as np

# mu3_sim 若已实现，取消注释:
# from month04.mu3_sim import find_similar

# 本节自包含版本：直接用 numpy 实现相似度检索
np.random.seed(42)


## 1. 用户 Profile = mean(liked song embeddings)

给定用户喜欢的歌曲 id 列表 `history = [i₀, i₁, ..., iₙ]`，从全量 embedding 矩阵 `all_embs`（形状 `[N, d]`）取出对应行，求平均：

```
user_emb = all_embs[history].mean(axis=0)   # shape: (d,)
```

然后 L2 归一化，使其模长为 1，以便用点积代替余弦相似度：

```
user_emb = user_emb / (np.linalg.norm(user_emb) + 1e-8)
```

这等价于协同过滤中"用户对喜欢的物品取平均隐向量"的做法（Item-based CF 的对偶视角）。embedding 质量越高（同类歌曲聚集越紧），平均向量就越能代表用户真实口味。


In [ ]:
# 演示：3首歌的 embedding 平均
N, d = 100, 8
all_embs = np.random.randn(N, d)
# 归一化所有 embedding（模拟已归一化的 song embeddings）
all_embs = all_embs / (np.linalg.norm(all_embs, axis=1, keepdims=True) + 1e-8)

history = [0, 5, 12]
user_emb_raw = all_embs[history].mean(axis=0)
user_emb = user_emb_raw / (np.linalg.norm(user_emb_raw) + 1e-8)
print(f"user_emb shape : {user_emb.shape}")
print(f"user_emb norm  : {np.linalg.norm(user_emb):.4f}  (应 ≈ 1.0)")
print(f"user_emb[:4]   : {user_emb[:4].round(4)}")


## 2. 冷启动问题：新用户没有历史怎么办？

当 `history = []` 时，无法计算平均 embedding。两种常见应对策略：

**策略 A — 流行度排名**：预先计算全库每首歌的播放次数，冷启动时返回全局 Top-K。公式上等价于用全量 embedding 的全局均值作为 user_emb（或直接按 popularity score 排序）：

```
# 全局平均（几何中心）作为冷启动 user_emb
cold_emb = all_embs.mean(axis=0)
cold_emb = cold_emb / (np.linalg.norm(cold_emb) + 1e-8)
```

**策略 B — 随机多样化**：随机采样若干歌曲作为初始推荐，降低马太效应，同时收集第一批交互信号。

策略 A 可解释性强；策略 B 探索性强。工业界通常先 A 再 B 混合（`ε-greedy`）。


In [ ]:
# 演示：冷启动 — 全局均值 vs 随机采样
cold_emb_global = all_embs.mean(axis=0)
cold_emb_global /= np.linalg.norm(cold_emb_global) + 1e-8

scores_cold = all_embs @ cold_emb_global
top_cold = np.argsort(scores_cold)[::-1][:5]
print(f"冷启动 Top-5（全局均值策略）: {top_cold.tolist()}")

# 随机多样化策略
top_random = np.random.choice(N, size=5, replace=False)
print(f"冷启动 Top-5（随机策略）    : {top_random.tolist()}")


## 3. 过滤已听：推荐结果去掉已喜欢的歌

排序后的候选列表中，历史歌曲得分必然偏高（它们定义了 user_emb），直接推荐会造成重复——用户已经知道这首歌，推给他毫无价值。

过滤方法：把 `history` 转成集合，在取 Top-K 时跳过集合内的 id：

```
history_set = set(user_history_ids)
ranked = np.argsort(scores)[::-1]
recs = [i for i in ranked if i not in history_set][:top_k]
```

时间复杂度：`O(N)` 排序 + `O(K)` 过滤，对百万量级需换用 FAISS + 后处理。Aurora 当前规模（万级歌曲）直接用 numpy 排序即可。


In [ ]:
# 演示：过滤前 vs 过滤后
scores_demo = all_embs @ user_emb
ranked_all = np.argsort(scores_demo)[::-1][:8]
print(f"过滤前 Top-8: {ranked_all.tolist()}")
print(f"history ids : {history}")

history_set = set(history)
ranked_filtered = [i for i in np.argsort(scores_demo)[::-1] if i not in history_set][:8]
print(f"过滤后 Top-8: {ranked_filtered}")
print(f"（历史歌曲 {[i for i in ranked_all if i in history_set]} 已被移除）")


## 4. ✏️ 实现 `recommend(user_history_ids, all_embs, top_k=10)`

**推理路线**：
1. 若 `user_history_ids` 为空，使用全量 embedding 全局均值作为冷启动 `user_emb`
2. 否则：`user_emb = all_embs[user_history_ids].mean(axis=0)`，再 L2 归一化
3. `scores = all_embs @ user_emb`（向量化余弦相似度，`O(N·d)`）
4. 按 `scores` 降序排列，跳过 `user_history_ids` 中的 id，取前 `top_k` 个

**参考输入输出**：
```
N, d = 200, 16
all_embs = ... # 200首歌，每首16维归一化 embedding
# 用户喜欢 id=[0,1,2,3,4]（假设这5首属于同一风格簇）
recs = recommend([0,1,2,3,4], all_embs, top_k=5)
# 期望：recs 是长度5的 int 列表，不含 0-4 中任何元素
# 且 recs 里的歌与该风格簇余弦相似度 > 全库均值
```

<details><summary>点击查看参考实现</summary>

```python
def recommend(user_history_ids, all_embs, top_k=10):
    """
    user_history_ids : list[int]   用户喜欢的歌曲 id
    all_embs         : np.ndarray  shape (N, d)，L2 归一化的歌曲 embedding
    top_k            : int         返回条数
    返回             : list[int]   推荐歌曲 id（已过滤历史）
    """
    N = len(all_embs)
    if len(user_history_ids) == 0:
        # 冷启动：全局均值
        user_emb = all_embs.mean(axis=0)
    else:
        user_emb = all_embs[user_history_ids].mean(axis=0)

    norm = np.linalg.norm(user_emb)
    if norm > 1e-8:
        user_emb = user_emb / norm

    scores = all_embs @ user_emb          # (N,)
    ranked = np.argsort(scores)[::-1]     # 降序

    history_set = set(user_history_ids)
    recs = [int(i) for i in ranked if i not in history_set][:top_k]
    return recs
```
</details>


In [ ]:
def recommend(user_history_ids, all_embs, top_k=10):
    """
    user_history_ids : list[int]   用户喜欢的歌曲 id
    all_embs         : np.ndarray  shape (N, d)，L2 归一化的歌曲 embedding
    top_k            : int         返回条数
    返回             : list[int]   推荐歌曲 id（已过滤历史）
    """
    # ✏️ TODO 步骤1：处理冷启动（history 为空）→ 用全局均值
    # ✏️ TODO 步骤2：计算 user_emb = mean(all_embs[history])，L2 归一化
    # ✏️ TODO 步骤3：scores = all_embs @ user_emb
    # ✏️ TODO 步骤4：降序排列，过滤 history_set，取 top_k
    pass


In [ ]:
# ── 自动检查 ──────────────────────────────────────────────────────────────────
np.random.seed(0)
N_test, d_test = 50, 8

# 构造有聚类结构的 embedding：前10首歌属于"爵士簇"
jazz_center = np.array([1.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
embs_test = np.random.randn(N_test, d_test) * 0.3
embs_test[:10] += jazz_center          # 爵士簇：id 0-9
embs_test = embs_test / (np.linalg.norm(embs_test, axis=1, keepdims=True) + 1e-8)

# 用户喜欢5首爵士（id 0-4）
history_test = [0, 1, 2, 3, 4]
recs = recommend(history_test, embs_test, top_k=5)

assert recs is not None, "recommend() 返回了 None，请完成 TODO"
assert len(recs) == 5, f"期望5条推荐，得到 {len(recs)}"
assert all(isinstance(r, int) for r in recs), "推荐 id 应为 int"
assert not any(r in set(history_test) for r in recs), "推荐中不应含历史歌曲"

# 推荐结果应以爵士簇（id 5-9）为主
jazz_hits = sum(1 for r in recs if r < 10)
assert jazz_hits >= 3, f"期望推荐 ≥3 首爵士歌曲，实际 {jazz_hits}（embedding 聚类质量或实现有误）"

print(f"✅ recommend() 通过：推荐列表 = {recs}，爵士命中 {jazz_hits}/5")

# 冷启动检查
recs_cold = recommend([], embs_test, top_k=3)
assert recs_cold is not None and len(recs_cold) == 3, "冷启动应返回3条"
print(f"✅ 冷启动通过：{recs_cold}")


## 5. 参数实验：模拟 10 个用户，画 Precision@5 分布

**实验设计**：
- `n_users = 10`，每个用户随机选 `history_len ∈ [3, 10]` 首歌作为历史
- `top_k = 5`，推荐后统计 Precision@5：推荐中属于"同风格簇"的比例
- 控制变量：改变 **embedding 噪声系数** `noise_std`（0.1 → 0.5 → 1.0），  观察 Precision@5 随 embedding 质量下降的变化趋势

**预期现象**：
- `noise_std = 0.1`（高质量 embedding，簇紧密）：Precision@5 集中在 0.6–1.0
- `noise_std = 1.0`（低质量 embedding，簇散乱）：Precision@5 接近随机基线 `≈ 0.1–0.2`
- 历史越长（history_len 越大），平均 embedding 越稳定，Precision@5 方差越小


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def run_experiment(noise_std, n_users=10, n_songs=200, d=16, n_genres=5, top_k=5):
    """生成多风格 embedding，模拟用户，计算 Precision@5"""
    np.random.seed(42)
    songs_per_genre = n_songs // n_genres  # 每类歌曲数

    # 构造 embedding：每类歌曲围绕一个随机中心
    centers = np.random.randn(n_genres, d)
    centers /= np.linalg.norm(centers, axis=1, keepdims=True) + 1e-8
    embs = np.vstack([
        centers[g] + np.random.randn(songs_per_genre, d) * noise_std
        for g in range(n_genres)
    ])
    embs /= np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8

    # 每首歌的真实类别标签
    labels = np.repeat(np.arange(n_genres), songs_per_genre)

    precisions = []
    for u in range(n_users):
        # 随机选一个主类别作为该用户偏好
        fav_genre = np.random.randint(n_genres)
        genre_ids = np.where(labels == fav_genre)[0].tolist()

        hist_len = np.random.randint(3, min(11, len(genre_ids)))
        history = np.random.choice(genre_ids, size=hist_len, replace=False).tolist()

        recs = recommend(history, embs, top_k=top_k)
        if recs is None:
            precisions.append(0.0)
            continue
        hits = sum(1 for r in recs if labels[r] == fav_genre)
        precisions.append(hits / top_k)

    return precisions

noise_levels = [0.1, 0.5, 1.0]
labels_plot  = ["高质量 (noise=0.1)", "中等 (noise=0.5)", "低质量 (noise=1.0)"]
colors       = ["#2ecc71", "#f39c12", "#e74c3c"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
fig.suptitle("Precision@5 分布 vs Embedding 质量（10 用户模拟）", fontsize=13)

for ax, noise, label, color in zip(axes, noise_levels, labels_plot, colors):
    precs = run_experiment(noise_std=noise)
    ax.bar(range(len(precs)), precs, color=color, alpha=0.8)
    ax.axhline(np.mean(precs), color="black", linestyle="--", linewidth=1.2,
               label=f"mean={np.mean(precs):.2f}")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("用户编号")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Precision@5")
plt.tight_layout()
plt.savefig("mu4_precision_experiment.png", dpi=100, bbox_inches="tight")
plt.show()
print("✅ 实验图已保存至 mu4_precision_experiment.png")
print()

# 打印均值汇总
for noise, label in zip(noise_levels, labels_plot):
    precs = run_experiment(noise_std=noise)
    print(f"{label}: mean Precision@5 = {np.mean(precs):.3f}  std = {np.std(precs):.3f}")


## 本课收束

本节实现了 `recommend(user_history_ids, all_embs, top_k)`，输出为过滤已听歌曲后的推荐 id 列表（`list[int]`，长度 `top_k`）。该函数对应 Aurora Music Intelligence Engine 的 `src/aurora/music/recommend.py`，与 `embed.py`（生成 `all_embs`）和 `sim.py`（可替换 numpy 排序为 FAISS）协同工作。下一节（`mu5_eval`）将在更大规模数据集上系统评估 Recall@K 与 NDCG，并对比 popularity baseline 与 embedding 推荐的差距。
